# Attention Heads

**Inside the Qwen3.6-35B-A3B Transformer**

This notebook steps through the attention mechanism in a real, running transformer.
We load Qwen3.6-35B-A3B, feed it text, and pause inside each attention head —
watching Q, K, V take shape, seeing RoPE rotate them, and reading the attention
weights that decide which tokens the model "looks at."

Companion to the [Ep11 article](https://huggingface.co/blog/EXDai/attention-heads).

---

## Setup

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.models.qwen3_5_moe.modeling_qwen3_5_moe import (
    Qwen3_5MoeAttention,
    Qwen3_5MoeDecoderLayer,
    Qwen3_5MoeGatedDeltaNet,
    create_causal_mask,
)

print(f"PyTorch {torch.__version__}")
print(f"CUDA: {torch.cuda.get_device_name()}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
import os

MODEL_NAME = "Qwen/Qwen3.6-35B-A3B"

# Point to the existing model cache so we don't re-download 70 GB.
# On atom the model lives at ~/cache/hf/hub/ (used by vLLM).
CACHE_DIR = os.path.expanduser("~/cache/hf/hub")
if os.path.isdir(CACHE_DIR):
    os.environ["HF_HOME"] = CACHE_DIR
    os.environ["TRANSFORMERS_CACHE"] = CACHE_DIR
    print(f"Using cache: {CACHE_DIR}")
else:
    print(f"Cache dir not found at {CACHE_DIR}, using default")

# Quick existence check before downloading anything
snapshot_dir = os.path.join(CACHE_DIR, "models--Qwen--Qwen3.6-35B-A3B", "snapshots")
if os.path.isdir(snapshot_dir):
    size_gb = sum(
        os.path.getsize(os.path.join(dp, f))
        for dp, _, fns in os.walk(snapshot_dir) for f in fns
    ) / 1e9
    print(f"Model found locally: {size_gb:.1f} GB")
else:
    print(f"WARNING: Model not found at {snapshot_dir}. Will attempt download.")

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Vocab size: {tok.vocab_size:,}")

# Load model with eager attention so we can extract real attention weights.
# SDPA / flash attention don't materialize the full weight matrix.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)
model.eval()

# The text-only language model sits inside the vision wrapper
lm = model.model.language_model
cfg = lm.config
print(f"Hidden size:    {cfg.hidden_size}")
print(f"Num layers:     {cfg.num_hidden_layers}")
print(f"Num Q heads:    {cfg.num_attention_heads}")
print(f"Num KV heads:   {cfg.num_key_value_heads}")
print(f"Head dim:       {cfg.head_dim}")
print(f"GQA ratio:      {cfg.num_attention_heads // cfg.num_key_value_heads}:1")

In [ ]:
# Identify full-attention vs linear-attention layers
layer_types = cfg.layer_types
full_attn_layers = [i for i, t in enumerate(layer_types) if t == "full_attention"]
linear_attn_layers = [i for i, t in enumerate(layer_types) if t == "linear_attention"]

print(f"Full attention layers ({len(full_attn_layers)}): {full_attn_layers}")
print(f"Linear attention layers ({len(linear_attn_layers)}): the other {len(linear_attn_layers)}")
print(f"Pattern: 3 linear → 1 full, repeating {len(full_attn_layers)}×")

# Show the first few layers with their module types
print(f"\nLayer modules:")
for i in range(min(12, len(layer_types))):
    layer = lm.layers[i]
    attn_name = type(layer.self_attn).__name__ if hasattr(layer, 'self_attn') else type(layer.linear_attn).__name__
    marker = " ◀ FULL" if layer_types[i] == "full_attention" else ""
    print(f"  Layer {i:2d}: {attn_name}{marker}")
print(f"  ...")

---

## Pick an Input Text

We'll use a short sentence with clear grammatical structure so attention patterns
are interpretable.

In [ ]:
text = "The cat sat on the mat because it was tired."

inputs = tok(text, return_tensors="pt").to(model.device)
token_ids = inputs["input_ids"][0]
tokens = [tok.decode(tid) for tid in token_ids]

print(f"Text:  {text}")
print(f"Tokens ({len(tokens)}): {tokens}")
print(f"IDs:   {token_ids.tolist()}")

---

## Forward Pass with Attention Capture

We run the model with `output_attentions=True`. The model's `capture_outputs`
decorator intercepts attention weights from each `Qwen3_5MoeAttention` module.

Linear-attention layers (Gated DeltaNet) don't produce explicit attention
weights — they use a kernelized recurrence, so they'll return `None`.

Each attention tensor shape: `(batch, num_heads, seq_len, seq_len)`.

In [ ]:
with torch.no_grad():
    outputs = model(**inputs, output_attentions=True)

attentions = outputs.attentions  # tuple of tensors or Nones per layer
print(f"Got {len(attentions)} attention entries (one per layer)")

# Show which layers have real attention weights
for i, attn in enumerate(attentions):
    if attn is not None:
        print(f"  Layer {i:2d} ({layer_types[i]}): shape {attn.shape} — {attn.shape[1]} heads × {attn.shape[2]}×{attn.shape[3]}")
    else:
        print(f"  Layer {i:2d} ({layer_types[i]}): None (no explicit weights)")

---

## Attention Weight Heatmaps

For each full-attention layer, we plot the mean attention weights across all
16 heads. Each cell $(i,j)$ shows how much token $j$ is attended to when
processing token $i$. The causal mask ensures token $i$ can only see tokens
$\le i$ (upper triangle is zero).

In [ ]:
seq_len = len(tokens)

fig, axes = plt.subplots(len(full_attn_layers), 1, figsize=(14, 3.2 * len(full_attn_layers)))
if len(full_attn_layers) == 1:
    axes = [axes]

for ax, layer_idx in zip(axes, full_attn_layers):
    attn = attentions[layer_idx][0].float()  # (heads, seq, seq)
    mean_attn = attn.mean(dim=0).cpu().numpy()  # average across heads

    im = ax.imshow(mean_attn, cmap="YlOrRd", aspect="auto", vmin=0, vmax=mean_attn.max())
    ax.set_xticks(range(seq_len))
    ax.set_xticklabels(tokens, rotation=45, ha="right", fontsize=9)
    ax.set_yticks(range(seq_len))
    ax.set_yticklabels(tokens, fontsize=9)
    ax.set_ylabel(f"Layer {layer_idx}", fontweight="bold")
    ax.set_xlabel("Attended token (key)")
    plt.colorbar(im, ax=ax, label="Attention weight")

fig.suptitle("Mean Attention Weights per Full-Attention Layer", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---

## Head-by-Head View

Each of the 16 heads learns a different attention pattern. Let's inspect them
individually at the first full-attention layer.

In [ ]:
first_full = full_attn_layers[0]
attn_first = attentions[first_full][0].float().cpu().numpy()  # (heads, seq, seq)
num_heads = attn_first.shape[0]

fig, axes = plt.subplots(4, 4, figsize=(16, 14))
for h in range(num_heads):
    ax = axes[h // 4][h % 4]
    im = ax.imshow(attn_first[h], cmap="YlOrRd", aspect="auto", vmin=0, vmax=attn_first[h].max())
    ax.set_xticks(range(seq_len))
    ax.set_xticklabels(tokens, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(seq_len))
    ax.set_yticklabels(tokens, fontsize=8)
    ax.set_title(f"Head {h}", fontsize=10)
    plt.colorbar(im, ax=ax)

fig.suptitle(f"All 16 Heads — Layer {first_full} (Full Attention)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

**What to notice:**
- **Diagonal-dominant heads**: attend mostly to the current token ("don't look around")
- **Previous-token heads**: strongest signal on the sub-diagonal (local syntax)
- **Punctuation heads**: some heads spike on periods, commas
- **Broad-context heads**: spread weight across many previous tokens
- Despite all 16 heads sharing only 2 KV heads (GQA 8:1), they learn distinct patterns

---

## Stepping Through a Single Attention Head

The heatmaps above show the *result* of attention. Now we break open the
machinery and watch each step:

1. **Q/K/V projection** — hidden states → query, key, value
2. **QK Norm** — RMSNorm applied to Q and K
3. **RoPE** — rotate Q and K by position-dependent angles
4. **GQA expansion** — repeat KV heads to match Q head count
5. **Scores** — $QK^T / \sqrt{d_k}$
6. **Causal mask** — hide future tokens
7. **Softmax** — convert to probabilities
8. **Weighted sum** — attention weights × V
9. **Gating** — sigmoid gate modulates output
10. **Output projection** — back to hidden_size

We'll run hidden states up to the target layer, then manually execute
each step with the actual model weights.

In [ ]:
target_layer = full_attn_layers[0]  # first full-attention layer
attn_module: Qwen3_5MoeAttention = lm.layers[target_layer].self_attn

head_dim = cfg.head_dim
num_q_heads = cfg.num_attention_heads
num_kv_heads = cfg.num_key_value_heads

print(f"Target: layer {target_layer}")
print(f"Module:  {attn_module.__class__.__name__}")
print(f"Q proj:  {attn_module.q_proj.in_features} → {attn_module.q_proj.out_features}  (×2 for gate)")
print(f"K proj:  {attn_module.k_proj.in_features} → {attn_module.k_proj.out_features}")
print(f"V proj:  {attn_module.v_proj.in_features} → {attn_module.v_proj.out_features}")
print(f"O proj:  {attn_module.o_proj.in_features} → {attn_module.o_proj.out_features}")
print(f"Q norm:  {attn_module.q_norm}")
print(f"K norm:  {attn_module.k_norm}")

In [ ]:
# Get embeddings and position encodings
with torch.no_grad():
    embed = lm.embed_tokens(inputs["input_ids"])
    print(f"Embedding shape: {embed.shape}  (batch, seq, hidden={cfg.hidden_size})")

    # Compute position IDs (4D for mRoPE: [T, H, W, batch×seq])
    past_seen_tokens = 0
    pos = torch.arange(embed.shape[1], device=embed.device) + past_seen_tokens
    base_pos_ids = pos.view(1, 1, -1).expand(4, embed.shape[0], -1)
    text_pos_ids = base_pos_ids[0]  # text positions: (batch, seq)

    # Compute RoPE cos/sin
    rope_cos, rope_sin = lm.rotary_emb(embed, base_pos_ids)
    print(f"RoPE cos: {rope_cos.shape}, sin: {rope_sin.shape}")

In [ ]:
# Run through decoder layers up to (but not including) the target layer.
# Each decoder layer transforms hidden states through its attention/mlp block.

# Build causal mask for full-attention layers
causal_mask = create_causal_mask(
    config=cfg,
    inputs_embeds=embed,
    attention_mask=inputs.get("attention_mask", None),
    past_key_values=None,
    position_ids=text_pos_ids,
)

# Build linear attention mask for the linear layers
linear_mask = lm._update_linear_attn_mask(
    inputs.get("attention_mask", None), past_key_values=None
)

hidden_states = embed
with torch.no_grad():
    for i in range(target_layer):
        layer = lm.layers[i]
        layer_mask = linear_mask if layer_types[i] == "linear_attention" else causal_mask

        hidden_states = layer(
            hidden_states,
            position_embeddings=(rope_cos, rope_sin),
            attention_mask=layer_mask,
            position_ids=text_pos_ids,
        )

print(f"Hidden states at layer {target_layer} input:")
print(f"  Shape: {hidden_states.shape}")
print(f"  dtype: {hidden_states.dtype}, device: {hidden_states.device}")
print(f"  Norm per token: {hidden_states.norm(dim=-1).squeeze(0).tolist()}")

---

### Step 1: Q, K, V Projections

The hidden states hit three weight matrices simultaneously. Note the Q
projection outputs `num_heads × head_dim × 2` — the factor of 2 is for
the **gate** (Qwen3.5 uses gated attention).

In [ ]:
seq_len = hidden_states.shape[1]

with torch.no_grad():
    # Q projection (includes gate): hidden → (16 × 256 × 2) = 8192
    q_proj_out = attn_module.q_proj(hidden_states)
    q_proj_out = q_proj_out.view(1, seq_len, num_q_heads, head_dim * 2)
    query_states, gate = torch.chunk(q_proj_out, 2, dim=-1)
    # query_states: (1, seq, 16, 256), gate: same

    # K projection: hidden → (2 × 256) = 512
    key_states = attn_module.k_proj(hidden_states)
    key_states = key_states.view(1, seq_len, num_kv_heads, head_dim)

    # V projection: hidden → 512
    value_states = attn_module.v_proj(hidden_states)
    value_states = value_states.view(1, seq_len, num_kv_heads, head_dim)

print(f"Query:  {query_states.shape}  ({num_q_heads} heads)")
print(f"Key:    {key_states.shape}  ({num_kv_heads} heads — GQA)")
print(f"Value:  {value_states.shape}  ({num_kv_heads} heads)")
print(f"Gate:   {gate.shape}  (pre-sigmoid)")

---

### Step 2: QK Norm

Qwen3.5 applies RMSNorm to Q and K *before* RoPE and attention.
This is a departure from the original Transformer and helps stabilize
training at scale. V is not normalized.

In [ ]:
with torch.no_grad():
    query_states = attn_module.q_norm(query_states)
    key_states = attn_module.k_norm(key_states)

# Show the effect: norms before and after
print(f"After QK Norm:")
print(f"  Q norm per head: {query_states.norm(dim=-1).squeeze(0)[:, 0].tolist()}")
print(f"  K norm per head: {key_states.norm(dim=-1).squeeze(0)[:, 0].tolist()}")

---

### Step 3: Apply RoPE

Q and K are rotated by position-dependent angles. Only the first 64 of
256 dimensions are rotated (25% partial rotary factor). The remaining
192 are *position-free* — they carry pure semantic content.

This is the same mRoPE mechanism from Ep10, now happening live inside
the attention layer.

In [ ]:
from transformers.models.qwen3_5_moe.modeling_qwen3_5_moe import apply_rotary_pos_emb

with torch.no_grad():
    # Transpose to (batch, heads, seq, head_dim)
    q_for_rope = query_states.transpose(1, 2)
    k_for_rope = key_states.transpose(1, 2)

    q_rotated, k_rotated = apply_rotary_pos_emb(q_for_rope, k_for_rope, rope_cos, rope_sin)

print(f"Q rotated: {q_rotated.shape}")
print(f"K rotated: {k_rotated.shape}")

# Verify rotation only affected the rotary dimensions
rotary_dims = int(head_dim * 0.25)  # 64
diff_rotary = (q_for_rope[:, :, :, :rotary_dims] - q_rotated[:, :, :, :rotary_dims]).abs().mean().item()
diff_pass = (q_for_rope[:, :, :, rotary_dims:] - q_rotated[:, :, :, rotary_dims:]).abs().mean().item()
print(f"\nMean |Δ| in rotary dims (first {rotary_dims}):  {diff_rotary:.6f}  ← rotated")
print(f"Mean |Δ| in pass-through dims (last {head_dim - rotary_dims}): {diff_pass:.8f}  ← untouched")

---

### Step 4: Attention Scores

The core operation: $\text{scores} = \frac{QK^T}{\sqrt{d_k}}$

First, we expand the 2 KV heads to match the 16 Q heads (GQA):
each KV head is repeated 8 times.

In [ ]:
# GQA expansion: repeat KV heads
n_rep = num_q_heads // num_kv_heads  # 8

# k_rotated: (1, 2, seq, 256) → (1, 16, seq, 256)
k_expanded = k_rotated.unsqueeze(2).expand(-1, -1, n_rep, -1, -1)
k_expanded = k_expanded.reshape(1, num_q_heads, seq_len, head_dim)

v_for_attn = value_states.transpose(1, 2)  # (1, 2, seq, 256)
v_expanded = v_for_attn.unsqueeze(2).expand(-1, -1, n_rep, -1, -1)
v_expanded = v_expanded.reshape(1, num_q_heads, seq_len, head_dim)

print(f"K expanded: {k_expanded.shape}  (batch, Q_heads={num_q_heads}, seq, head_dim)")
print(f"V expanded: {v_expanded.shape}")

# QK^T / sqrt(d_k)
scaling = head_dim ** -0.5  # 1/√256 = 1/16 = 0.0625
attn_scores = torch.matmul(q_rotated, k_expanded.transpose(-2, -1)) * scaling

print(f"\nAttention scores: {attn_scores.shape}  (batch, heads, query_seq, key_seq)")
print(f"Score range: [{attn_scores.min().item():.2f}, {attn_scores.max().item():.2f}]")

---

### Step 5: Causal Mask

Each token can only attend to itself and previous tokens.
Future positions are set to $-\infty$ (zero after softmax).

In [ ]:
# Create causal mask: lower triangular = 1, upper = 0
causal = torch.tril(torch.ones(seq_len, seq_len, device=attn_scores.device))
causal_expanded = causal[None, None, :, :]  # (1, 1, seq, seq)

masked_scores = attn_scores.masked_fill(causal_expanded == 0, float("-inf"))

# Visualize the mask
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.imshow(causal_expanded[0, 0].cpu(), cmap="Greys", aspect="auto")
ax1.set_title("Causal Mask (white = visible, black = masked)")
ax1.set_xticks(range(seq_len)); ax1.set_xticklabels(tokens, rotation=45, ha="right", fontsize=8)
ax1.set_yticks(range(seq_len)); ax1.set_yticklabels(tokens, fontsize=8)

# Show masked scores for head 0
ax2.imshow(masked_scores[0, 0].float().cpu(), cmap="coolwarm", aspect="auto")
ax2.set_title("Masked Scores — Head 0 (blue = -inf, red = positive)")
ax2.set_xticks(range(seq_len)); ax2.set_xticklabels(tokens, rotation=45, ha="right", fontsize=8)
ax2.set_yticks(range(seq_len)); ax2.set_yticklabels(tokens, fontsize=8)
plt.tight_layout()
plt.show()

---

### Step 6: Softmax

Converts scores to probabilities. Each row sums to 1.
The `-inf` entries become 0.

In [ ]:
attn_weights = F.softmax(masked_scores, dim=-1, dtype=torch.float32).to(q_rotated.dtype)

print(f"Attention weights: {attn_weights.shape}")
print(f"Weight range: [{attn_weights.min().item():.6f}, {attn_weights.max().item():.6f}]")

# Verify each row sums to 1
row_sums = attn_weights[0, 0].sum(dim=-1)  # head 0
print(f"Row sums for head 0 — min: {row_sums.min().item():.6f}, max: {row_sums.max().item():.6f}")

---

### Step 7: Weighted Sum + Gate + Output Projection

Three final operations:

1. **Weighted sum**: attention weights × V
2. **Gating**: multiply by $\sigma(\text{gate})$ — the model's "trust" knob
3. **Output projection**: linear map back to hidden_size

In [ ]:
# 1. Weighted sum: attn_weights @ V
attn_output_per_head = torch.matmul(attn_weights, v_expanded)
print(f"Per-head output: {attn_output_per_head.shape}")

# Concatenate heads: (1, heads, seq, head_dim) → (1, seq, heads×head_dim)
attn_output_concat = (
    attn_output_per_head.transpose(1, 2).contiguous()
    .reshape(1, seq_len, num_q_heads * head_dim)
)
print(f"Concatenated: {attn_output_concat.shape}")

# 2. Gating: sigmoid of the gate projection
gate_weight = torch.sigmoid(gate).reshape(1, seq_len, num_q_heads * head_dim)
gated_output = attn_output_concat * gate_weight
print(f"After gating: {gated_output.shape}")
print(f"Gate activation — mean: {gate_weight.mean().item():.3f}, std: {gate_weight.std().item():.3f}")

# 3. Output projection: back to hidden_size
final_output = attn_module.o_proj(gated_output)
print(f"Final output: {final_output.shape}  (batch, seq, hidden_size={cfg.hidden_size})")

---

## Verify: Manual vs Model Attention

Our manually computed attention weights should match what the model
returned via `output_attentions=True`.

In [ ]:
model_attn = attentions[target_layer][0].float()  # (heads, seq, seq) from model
manual_attn = attn_weights[0].float()  # (heads, seq, seq) our computation

diff = (model_attn - manual_attn).abs()
print(f"Max difference:  {diff.max().item():.8f}")
print(f"Mean difference: {diff.mean().item():.8f}")

if diff.max().item() < 1e-4:
    print("\n✓ Manual computation matches model exactly.")
else:
    print("\n⚠ Small differences exist (likely due to mask handling differences)")

# Side by side for first head
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

h = 0
ax1.imshow(model_attn[h].cpu(), cmap="YlOrRd", aspect="auto")
ax1.set_title(f"Model — Head {h}")
ax1.set_xticks(range(seq_len)); ax1.set_xticklabels(tokens, rotation=45, ha="right", fontsize=8)
ax1.set_yticks(range(seq_len)); ax1.set_yticklabels(tokens, fontsize=8)

ax2.imshow(manual_attn[h].cpu(), cmap="YlOrRd", aspect="auto")
ax2.set_title(f"Manual — Head {h}")
ax2.set_xticks(range(seq_len)); ax2.set_xticklabels(tokens, rotation=45, ha="right", fontsize=8)
ax2.set_yticks(range(seq_len)); ax2.set_yticklabels(tokens, fontsize=8)

ax3.imshow(diff[h].cpu(), cmap="Reds", aspect="auto")
ax3.set_title(f"|Difference| — Head {h}")
ax3.set_xticks(range(seq_len)); ax3.set_xticklabels(tokens, rotation=45, ha="right", fontsize=8)
ax3.set_yticks(range(seq_len)); ax3.set_yticklabels(tokens, fontsize=8)

plt.tight_layout()
plt.show()

---

## GQA: Grouped Query Attention

With 16 Q heads sharing only 2 KV heads, each KV head serves 8 Q heads.
Heads 0–7 share KV head 0; heads 8–15 share KV head 1.

Despite sharing the same K and V, the Q heads learn different attention
patterns — the variation comes from different Q projections.

In [ ]:
print(f"GQA group size: {n_rep} Q heads per KV head")
print(f"KV head 0 → Q heads {list(range(0, 8))}")
print(f"KV head 1 → Q heads {list(range(8, 16))}")

# Compare attention patterns grouped by KV head
fig, axes = plt.subplots(2, 8, figsize=(22, 7))
for h in range(num_q_heads):
    kv_head = h // n_rep
    ax = axes[kv_head][h % 8]
    im = ax.imshow(manual_attn[h].cpu(), cmap="YlOrRd", aspect="auto")
    ax.set_title(f"Q head {h} (KV head {kv_head})", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])

fig.suptitle("Attention Patterns Grouped by Shared KV Head", fontsize=14)
plt.tight_layout()
plt.show()

# Bonus: show the actual K and V from each KV head
with torch.no_grad():
    k0 = k_rotated[0, 0].float().cpu()  # KV head 0: (seq, head_dim)
    k1 = k_rotated[0, 1].float().cpu()  # KV head 1: (seq, head_dim)

print(f"\nKV head 0 activation norm per position: {k0.norm(dim=-1).tolist()}")
print(f"KV head 1 activation norm per position: {k1.norm(dim=-1).tolist()}")

---

## Attention Evolution Across Layers

How do attention patterns change as we go deeper?
We compare the mean attention pattern of each full-attention layer.

In [ ]:
fig, axes = plt.subplots(len(full_attn_layers), 1, figsize=(14, 3 * len(full_attn_layers)))
if len(full_attn_layers) == 1:
    axes = [axes]

for ax, layer_idx in zip(axes, full_attn_layers):
    attn = attentions[layer_idx][0].float().cpu().numpy()
    mean_attn = attn.mean(axis=0)  # average over heads

    # For each query position (first 8, for clarity), show attention distribution
    colors = plt.cm.tab10(np.linspace(0, 1, min(8, seq_len)))
    for pos in range(min(8, seq_len)):
        ax.plot(range(seq_len), mean_attn[pos], color=colors[pos],
                marker='o', markersize=4, label=f'"{tokens[pos]}"', alpha=0.8)

    ax.set_xticks(range(seq_len))
    ax.set_xticklabels(tokens, rotation=45, ha="right", fontsize=9)
    ax.set_ylabel(f"Layer {layer_idx}\nMean attn weight")
    ax.legend(fontsize=8, loc="upper right", ncol=2)
    ax.grid(True, alpha=0.3)

fig.suptitle("Attention Distribution per Query Position — All Full-Attention Layers", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---

## The Gating Mechanism

Qwen3.5 uses **gated attention**: $\text{output} = \text{softmax}(QK^T/\sqrt{d})V \odot \sigma(\text{gate})$

The gate is learned — the model decides per-head, per-token, per-dimension
how much to trust the attention output. A gate near 0 silences the head;
near 1 passes it through unchanged.

In [ ]:
# Gate values we captured earlier
gate_values = torch.sigmoid(gate).squeeze(0)  # (seq, heads, head_dim)

# Mean gate per head
mean_gate_per_head = gate_values.mean(dim=[0, 2]).cpu().numpy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.bar(range(num_q_heads), mean_gate_per_head, color='steelblue')
ax1.set_xlabel("Head")
ax1.set_ylabel("Mean σ(gate)")
ax1.set_title(f"Mean Gate Activation per Head — Layer {target_layer}")
ax1.axhline(y=0.5, color="red", linestyle="--", alpha=0.5, label="σ=0.5")
ax1.legend()
ax1.set_ylim(0, 1)

# Gate value per token per head (mean over dim)
gate_per_token_head = gate_values.mean(dim=2).cpu().numpy()  # (seq, heads)
im = ax2.imshow(gate_per_token_head.T, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)
ax2.set_xticks(range(seq_len))
ax2.set_xticklabels(tokens, rotation=45, ha="right", fontsize=9)
ax2.set_yticks(range(num_q_heads))
ax2.set_ylabel("Head")
ax2.set_xlabel("Token")
ax2.set_title(f"Gate Value per Token × Head — Layer {target_layer}")
plt.colorbar(im, ax=ax2, label="σ(gate)")

plt.tight_layout()
plt.show()

---

## Attention Pattern: Pronoun Resolution

A classic test: does "it" attend to "cat"? In our sentence
"The cat sat on the mat because it was tired," the word "it"
should look back at "cat."

Let's check which heads are doing this resolution.

In [ ]:
# Find positions of "it" and "cat"
it_positions = [i for i, t in enumerate(tokens) if "it" in t.lower()]
cat_positions = [i for i, t in enumerate(tokens) if "cat" in t.lower()]

print(f"'cat' positions: {cat_positions} → {[tokens[p] for p in cat_positions]}")
print(f"'it' positions:  {it_positions} → {[tokens[p] for p in it_positions]}")

if it_positions and cat_positions:
    it_pos = it_positions[0]
    cat_pos = cat_positions[0]

    # For each full-attention layer, show how much "it" attends to "cat"
    print(f"\nAttention from 'it' (pos {it_pos}) → 'cat' (pos {cat_pos}):")
    for layer_idx in full_attn_layers:
        attn = attentions[layer_idx][0].float()  # (heads, seq, seq)
        it_to_cat = attn[:, it_pos, cat_pos].cpu().numpy()
        top_heads = np.argsort(it_to_cat)[::-1][:3]
        print(f"  Layer {layer_idx:2d}: {it_to_cat.mean():.4f} (mean) — top heads: "
              f"{top_heads.tolist()} with values {it_to_cat[top_heads].tolist()}")

    # Show all attention from "it" across heads at the deepest full-attention layer
    deepest_full = full_attn_layers[-1]
    attn_deep = attentions[deepest_full][0].float()  # (heads, seq, seq)
    it_attn = attn_deep[:, it_pos, :].cpu().numpy()  # (heads, seq)

    fig, axes = plt.subplots(4, 4, figsize=(16, 14))
    for h in range(num_heads):
        ax = axes[h // 4][h % 4]
        bars = ax.bar(range(seq_len), it_attn[h], color='steelblue')
        bars[cat_pos].set_color('red')  # highlight 'cat'
        bars[it_pos].set_color('orange')  # highlight 'it' itself
        ax.set_xticks(range(seq_len))
        ax.set_xticklabels(tokens, rotation=45, ha="right", fontsize=8)
        ax.set_title(f"Head {h}", fontsize=10)
        ax.set_ylim(0, it_attn[h].max() * 1.1)

    fig.suptitle(f"What 'it' Attends To — Layer {deepest_full} (Deepest Full Attention)", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

---

## Full Attention vs Gated DeltaNet

36 of 40 layers use `Qwen3_5MoeGatedDeltaNet` — a gated linear attention
variant that avoids the $O(n^2)$ softmax. Only 4 layers use standard
softmax attention as "synchronization points" every 4th position.

This hybrid design is how Qwen3.5 stays efficient at 35B parameters
while keeping the expressivity of full attention where it matters most.

In [ ]:
# Compare the two attention modules
linear_layer_idx = linear_attn_layers[0]
delta_module = lm.layers[linear_layer_idx].linear_attn

print(f"Full attention (layer {target_layer}):")
print(f"  Class:    {attn_module.__class__.__name__}")
print(f"  Q proj:   {attn_module.q_proj.in_features} → {attn_module.q_proj.out_features}")
print(f"  K proj:   {attn_module.k_proj.in_features} → {attn_module.k_proj.out_features}")
print(f"  V proj:   {attn_module.v_proj.in_features} → {attn_module.v_proj.out_features}")
print(f"  O proj:   {attn_module.o_proj.in_features} → {attn_module.o_proj.out_features}")
print(f"  QK norm:  Yes")
print(f"  Gated:    Yes (sigmoid gate from Q projection)")
print(f"  Heads:    {num_q_heads} Q / {num_kv_heads} KV")
print(f"  Cost:     O(seq²) — materializes {seq_len}×{seq_len} weight matrix")

print(f"\nGated DeltaNet (layer {linear_layer_idx}):")
print(f"  Class:    {delta_module.__class__.__name__}")
for name, param in delta_module.named_parameters():
    print(f"  {name}: {list(param.shape)}")
print(f"  Cost:     O(seq) — recurrent state, no explicit weight matrix")
print(f"  Note:     Does NOT use RoPE or softmax. Linear-time kernelized attention.")

# Show the full pattern
print(f"\nFull layer pattern (first 16 of {len(layer_types)}):")
for i in range(min(16, len(layer_types))):
    label = "█ FULL " if layer_types[i] == "full_attention" else "· delta"
    print(f"  Layer {i:2d}: {label}")

---

## What We Saw

We stepped through the attention mechanism of a real, running
Qwen3.6-35B-A3B model:

1. **Embedding** → 2048-dim vectors for each token
2. **Q/K/V projections** — Q expands to 4096 (16 heads × 256 dim, doubled for gate),
   K/V stay lean at 2 heads each (GQA 8:1)
3. **QK Norm** — RMSNorm applied to Q and K before attention
4. **mRoPE** — only 25% of dimensions rotated; the rest carry position-free semantics
5. **Attention scores** — $QK^T / \sqrt{d_k}$, masked for causality
6. **Softmax** — converts scores to probabilities
7. **Weighted sum** — attention weights × V
8. **Gating** — sigmoid(gate) modulates output per head per token per dimension
9. **Output projection** — back to hidden_size (2048)

We also saw:
- **GQA in action**: 8 Q heads share each KV head, saving 8× memory
- **Attention evolution**: patterns deepen and spread across layers
- **Pronoun resolution**: "it" → "cat" attention grows in deeper layers
- **Hybrid architecture**: only 10% of layers use standard softmax attention;
  the rest use efficient Gated DeltaNet

---

## Explore Further

- **Change the text** — try a sentence with different structure and see how
  attention patterns shift
- **Compare layers** — early vs deep full-attention layers
- **Longer sequences** — watch the causal mask and see attention spread
- **Gate analysis** — which heads does the model silence? When?
- **Translation invariance** — shift the input text and verify attention
  patterns stay the same (linking back to Ep10's mRoPE property)
- **Multiple texts** — compare attention for similar vs different sentences